In [1]:
print('hello')

hello


In [1]:
from P2Plibrary import P2P

BASE_URL = "https://api.binance.com" # production base url
KEY = "---"
SECRET = "---"

p2p = P2P()
p2p.client(key=KEY,secret=SECRET)

params = {}
data ={

# "adOrderNo": "20462599282021847040"
}
result = p2p.C2C_PAYMENTMETHOD_listAll(params=params,data=data)
content = result.content
print(content)

ModuleNotFoundError: No module named 'P2Plibrary'

In [6]:
import requests
import hmac
import hashlib
import time
from urllib.parse import urlencode
from dotenv import load_dotenv
import os

# ------------------------------------------------
# Load environment variables
# ------------------------------------------------
load_dotenv()

API_KEY = os.getenv("API_KEY")
SECRET_KEY = os.getenv("SECRET_KEY")
BASE_URL = os.getenv("BASE_URL", "https://api.binance.com")


# ------------------------------------------------
# Hashing & timestamp functions (as in PDF)
# ------------------------------------------------
def hashing(query_string):
    return hmac.new(
        SECRET_KEY.encode("utf-8"), query_string.encode("utf-8"), hashlib.sha256
    ).hexdigest()


def get_timestamp():
    return int(time.time() * 1000)


# ------------------------------------------------
# Dispatch request helper (same as PDF)
# ------------------------------------------------
def dispatch_request(http_method):
    session = requests.Session()
    session.headers.update({
        "Content-Type": "application/json;charset=utf-8",
        "X-MBX-APIKEY": API_KEY,
        "clientType": "WEB"   # Important: uppercase WEB
    })
    return {
        "GET": session.get,
        "DELETE": session.delete,
        "PUT": session.put,
        "POST": session.post,
    }.get(http_method, session.get)


# ------------------------------------------------
# Send signed request (exact Binance PDF structure)
# ------------------------------------------------
def send_signed_request(http_method, url_path, payload=None, dataLoad=None):
    if payload is None:
        payload = {}
    if dataLoad is None:
        dataLoad = {}

    query_string = urlencode(payload)
    query_string = query_string.replace("%27", "%22")

    if query_string:
        query_string = f"{query_string}&timestamp={get_timestamp()}"
    else:
        query_string = f"timestamp={get_timestamp()}"

    signature = hashing(query_string)
    print(signature)
    url = f"{BASE_URL}{url_path}?{query_string}&signature={signature}"

    params = {"url": url, "params": {}, "data": dataLoad}
    response = dispatch_request(http_method)(**params)
    return response


# ------------------------------------------------
# Example: get C2C ads list (works only for Merchant key)
# ------------------------------------------------
def get_ads_list(page=1, rows=50, asset="USDT", fiatUnit="VND", tradeType=None, advStatus=None):
    endpoint = "/sapi/v1/c2c/ads/listWithPagination"

    dataLoad = {
        "page": page,
        "rows": rows,
        "asset": asset,
        "fiatUnit": fiatUnit
    }
    if tradeType:
        dataLoad["tradeType"] = tradeType
    if advStatus is not None:
        dataLoad["advStatus"] = advStatus

    response = send_signed_request("POST", endpoint, {}, dataLoad)
    try:
        return response.json()
    except ValueError:
        return {"error": "invalid_json_response", "status_code": response.status_code, "text": response.text}


if __name__ == "__main__":
    print("🔍 Requesting Ads List ...")
    result = get_ads_list()
    print(result)


🔍 Requesting Ads List ...
63e2d7ab2a962f606269027b498e7f83e377b4e337a4d57763c3f270dc7da505
{'code': -1022, 'msg': 'Signature for this request is not valid.'}


In [3]:
import os
import time
import hmac
import hashlib
import requests
import json
from urllib.parse import urlencode
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("API_KEY")
SECRET_KEY = os.getenv("SECRET_KEY")
BASE_URL = os.getenv("BASE_URL", "https://api.binance.com")


def get_timestamp():
    return int(time.time() * 1000)


def sign(query_string: str) -> str:
    return hmac.new(
        SECRET_KEY.encode("utf-8"),
        query_string.encode("utf-8"),
        hashlib.sha256
    ).hexdigest()


def send_signed_post(endpoint: str, body: dict):
    timestamp = get_timestamp()
    query_string = f"timestamp={timestamp}"
    signature = sign(query_string)

    url = f"{BASE_URL}{endpoint}?{query_string}&signature={signature}"

    headers = {
        "X-MBX-APIKEY": API_KEY,
        "Content-Type": "application/json",
        "clientType": "WEB"
    }

    print("🔍 URL:", url)
    print("📦 Body:", json.dumps(body, indent=2))

    resp = requests.post(url, headers=headers, data=json.dumps(body))
    try:
        return resp.json()
    except ValueError:
        return {"error": "invalid_json_response", "status_code": resp.status_code, "text": resp.text}


def get_ads_list():
    endpoint = "/sapi/v1/c2c/ads/listWithPagination"

    body = {
        "page": 1,
        "rows": 20,
        "asset": "USDT",
        "fiatUnit": "VND",
        "tradeType": "SELL"
    }

    return send_signed_post(endpoint, body)


if __name__ == "__main__":
    result = get_ads_list()
    print("\n📄 Kết quả:")
    print(json.dumps(result, indent=2, ensure_ascii=False))


🔍 URL: https://api.binance.com/sapi/v1/c2c/ads/listWithPagination?timestamp=1762530945810&signature=5b1b245d619b04885f7f155dd6961657c99fda466531e2d894e449c29ba85ded
📦 Body: {
  "page": 1,
  "rows": 20,
  "asset": "USDT",
  "fiatUnit": "VND",
  "tradeType": "SELL"
}

📄 Kết quả:
{
  "code": "000000",
  "message": "success",
  "data": [
    {
      "advNo": "13796836708352905216",
      "classify": "profession",
      "tradeType": "SELL",
      "asset": "USDT",
      "fiatUnit": "VND",
      "advStatus": 3,
      "priceType": 1,
      "priceFloatingRatio": "100.00000000",
      "rateFloatingRatio": "0.00000000",
      "price": "27866",
      "initAmount": "2513628.28000000",
      "surplusAmount": "135.76",
      "maxSingleTransAmount": "1500000000",
      "minSingleTransAmount": "10000000",
      "buyerKycLimit": 1,
      "buyerRegDaysLimit": 90,
      "buyerBtcPositionLimit": "-1.00000000",
      "remarks": "Không yêu cầu xác minh mất thời gian\nCó thể sử chuyển khoản bằng tài khoản thứ

In [2]:
import os
import time
import hmac
import hashlib
import requests
import json
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("API_KEY")
SECRET_KEY = os.getenv("SECRET_KEY")
BASE_URL = os.getenv("BASE_URL", "https://api.binance.com")

# ------------------------ Utility ------------------------
def get_timestamp():
    return int(time.time() * 1000)


def sign(query_string: str) -> str:
    """Tạo chữ ký HMAC-SHA256"""
    return hmac.new(
        SECRET_KEY.encode("utf-8"),
        query_string.encode("utf-8"),
        hashlib.sha256
    ).hexdigest()


def send_signed_post(endpoint: str, body: dict):
    """Gửi POST request có ký chữ ký"""
    timestamp = get_timestamp()
    query_string = f"timestamp={timestamp}"
    signature = sign(query_string)
    url = f"{BASE_URL}{endpoint}?{query_string}&signature={signature}"

    headers = {
        "X-MBX-APIKEY": API_KEY,
        "Content-Type": "application/json",
        "clientType": "WEB"
    }

    print("🔍 URL:", url)
    print("📦 Body:", json.dumps(body, indent=2, ensure_ascii=False))

    resp = requests.post(url, headers=headers, data=json.dumps(body))
    try:
        return resp.json()
    except ValueError:
        return {"error": "invalid_json_response", "status_code": resp.status_code, "text": resp.text}


# ------------------------ MAIN FUNCTION ------------------------
def set_ad_status(adv_no: str, enable: bool):
    """
    Bật/tắt quảng cáo.
    enable=True  => advStatus=1 (ON)
    enable=False => advStatus=2 (OFF)
    """
    endpoint = "/sapi/v1/c2c/ads/update"
    adv_status = 1 if enable else 2

    body = {
        "advNo": adv_no,
        "advStatus": adv_status,
        "updateMode": "selective"  # chỉ cập nhật trường advStatus
    }

    response = send_signed_post(endpoint, body)
    return response




In [4]:
if __name__ == "__main__":
    # ⚙️ Thay mã quảng cáo của bạn ở đây
    adv_no = "13796836591957819392"

    # 🟢 Bật quảng cáo
    print("🟢 Bật quảng cáo:")
    result_on = set_ad_status(adv_no, True)
    print(json.dumps(result_on, indent=2, ensure_ascii=False))




🟢 Bật quảng cáo:
🔍 URL: https://api.binance.com/sapi/v1/c2c/ads/update?timestamp=1762530977362&signature=83bd52cafb6e3cbfb6e85e0b06e6b598bc444122feba2282bc1e6e36a6eee927
📦 Body: {
  "advNo": "13796836591957819392",
  "advStatus": 1,
  "updateMode": "selective"
}
{
  "code": "000000",
  "message": "success",
  "data": true,
  "success": true
}
